In [7]:
import os
import pandas as pd

input_dir = '../Northwind Database - Dirty/dirty_data'

files = [
    'Categories.csv', 'CustomerCustomerDemo.csv', 'CustomerDemographics.csv',
    'Customers.csv', 'Employees.csv', 'EmployeeTerritories.csv',
    'Order_Details.csv', 'Orders.csv', 'Products.csv', 'Region.csv',
    'Shippers.csv', 'Suppliers.csv', 'Territories.csv'
]

# missingهای پنهان که باید NaN در نظر گرفته شوند
extra_na_values = ['', ' ', 'NA', 'N/A', 'NULL', 'null', 'None', '-', '?', '--']

print("="*70)
print("PHASE 3.2 - STEP 1 : DATA INSPECTION (REPORT ONLY)")
print("="*70)

for file in files:
    path = os.path.join(input_dir, file)
    print("\n" + "-"*70)
    print(f"FILE: {file}")
    print("-"*70)

    if not os.path.exists(path):
        print("  [!] File not found.")
        continue

    # خواندن با مدیریت encoding
    try:
        df = pd.read_csv(path, na_values=extra_na_values, keep_default_na=True)
    except UnicodeDecodeError:
        df = pd.read_csv(path, na_values=extra_na_values,
                         keep_default_na=True, encoding='latin-1')

    n_rows, n_cols = df.shape
    print(f"  Shape: {n_rows} rows x {n_cols} cols")

    # جدول خالی طبیعی (فقط سرستون)
    if n_rows == 0:
        print("  [i] Empty table (header only) - normal, not an error.")
        continue

    # ردیف‌های کاملاً تکراری
    dup_count = df.duplicated().sum()
    print(f"  Fully-duplicated rows: {dup_count}")

    # درصد missing هر ستون + علامت‌گذاری >30%
    miss_pct = (df.isna().mean() * 100).round(2)
    print("  Missing % per column:")
    for col, pct in miss_pct.items():
        flag = "  <-- OVER 30% (review)" if pct > 30 else ""
        print(f"    {col:<25} {pct:>6}%{flag}")

print("\n" + "="*70)
print("STEP 1 DONE - report only, nothing was modified or saved.")
print("="*70)


PHASE 3.2 - STEP 1 : DATA INSPECTION (REPORT ONLY)

----------------------------------------------------------------------
FILE: Categories.csv
----------------------------------------------------------------------
  Shape: 9 rows x 4 cols
  Fully-duplicated rows: 1
  Missing % per column:
    CategoryID                   0.0%
    CategoryName                 0.0%
    Description                44.44%  <-- OVER 30% (review)
    Picture                      0.0%

----------------------------------------------------------------------
FILE: CustomerCustomerDemo.csv
----------------------------------------------------------------------
  Shape: 0 rows x 2 cols
  [i] Empty table (header only) - normal, not an error.

----------------------------------------------------------------------
FILE: CustomerDemographics.csv
----------------------------------------------------------------------
  Shape: 0 rows x 2 cols
  [i] Empty table (header only) - normal, not an error.

-----------------------

In [13]:
import os
import re
import pandas as pd

input_dir = '../Northwind Database - Dirty/dirty_data'
output_dir = 'Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/cleaned_data'
os.makedirs(output_dir, exist_ok=True)

extra_na_values = ["NULL", "null", "NA", "N/A", "n/a", "-", "?", " ", ""]

# ستون‌هایی که missing آن‌ها معنادار است → NULL بماند
optional_text_columns = [
    'Region', 'ShipRegion', 'Fax', 'Notes', 'Photo', 'PhotoPath',
    'Description', 'HomePage', 'TitleOfCourtesy', 'Extension', 'ShippedDate'
]

# ستون‌های آدرس ارسال در Orders که می‌توان از ردیف‌های دیگرِ همان مشتری پر کرد
ship_smart_fill_cols = [
    'ShipName', 'ShipAddress', 'ShipCity',
    'ShipRegion', 'ShipPostalCode', 'ShipCountry'
]

def read_table(path):
    try:
        return pd.read_csv(path, na_values=extra_na_values, keep_default_na=True)
    except UnicodeDecodeError:
        return pd.read_csv(path, na_values=extra_na_values,
                           keep_default_na=True, encoding='latin-1')

smart_fill_report = []   # گزارش اثر Smart Fill
text_fill_report = []    # گزارش برخورد با بقیه متنی‌ها (Unknown یا NULL)

for fname in sorted(os.listdir(input_dir)):
    if not fname.lower().endswith('.csv'):
        continue

    path = os.path.join(input_dir, fname)
    df = read_table(path)

    text_cols = df.select_dtypes(include='object').columns

    # ---------- ۱) نرمال‌سازی متن ----------
    for col in text_cols:
        df[col] = (
            df[col].astype('string')
                   .str.strip()
                   .str.replace(r'\s+', ' ', regex=True)
        )
        df[col] = df[col].replace({'': pd.NA})

    # ---------- ۲) حذف duplicate ----------
    norm = df.copy()
    for col in text_cols:
        norm[col] = norm[col].astype('string').str.lower().str.strip()
    dup_mask = norm.duplicated()
    n_dup = int(dup_mask.sum())
    if n_dup > 0:
        df = df[~dup_mask].reset_index(drop=True)

    # ---------- ۳) Smart Fill فقط برای Orders ----------
    if fname.lower() == 'orders.csv' and 'CustomerID' in df.columns:
        for col in ship_smart_fill_cols:
            if col not in df.columns:
                continue
            before = int(df[col].isna().sum())
            df[col] = (
                df.groupby('CustomerID')[col]
                  .transform(lambda x: x.ffill().bfill())
            )
            after = int(df[col].isna().sum())
            smart_fill_report.append({
                'Table': fname,
                'Column': col,
                'Missing_Before': before,
                'Missing_After': after,
                'Recovered': before - after
            })

    # ---------- ۴) پرکردن متن و ثبت در گزارش ----------
    for col in text_cols:
        n_miss = int(df[col].isna().sum())
        if n_miss > 0:
            if col in optional_text_columns:
                # ثبت در گزارش: رها شده به عنوان NULL
                text_fill_report.append({
                    'Table': fname,
                    'Column': col,
                    'Missing_Count': n_miss,
                    'Action_Taken': 'Left as NULL (Optional)'
                })
            else:
                # ثبت در گزارش: پر شده با Unknown
                text_fill_report.append({
                    'Table': fname,
                    'Column': col,
                    'Missing_Count': n_miss,
                    'Action_Taken': "Filled with 'Unknown'"
                })
                df[col] = df[col].fillna('Unknown')

    # ---------- ۵) ذخیره ----------
    out_path = os.path.join(output_dir, fname)
    df.to_csv(out_path, index=False)
    print(f"[OK] {fname:25s} rows={len(df):5d}  dups_removed={n_dup}")


# ================= گزارش‌ها =================
print("\n" + "=" * 70)
print("1. SMART FILL EFFECT REPORT (Orders Table)")
print("=" * 70)
if smart_fill_report:
    rep_smart = pd.DataFrame(smart_fill_report)
    print(rep_smart.to_string(index=False))
else:
    print("No Smart Fill applied.")

print("\n" + "=" * 70)
print("2. TEXT IMPUTATION REPORT (Other Text Columns)")
print("=" * 70)
if text_fill_report:
    rep_text = pd.DataFrame(text_fill_report)
    print(rep_text.to_string(index=False))
else:
    print("No other missing text values found.")
print("=" * 70)

[OK] Categories.csv            rows=    8  dups_removed=1
[OK] CustomerCustomerDemo.csv  rows=    0  dups_removed=0
[OK] CustomerDemographics.csv  rows=    0  dups_removed=0
[OK] Customers.csv             rows=   91  dups_removed=2
[OK] EmployeeTerritories.csv   rows=   51  dups_removed=3
[OK] Employees.csv             rows=   10  dups_removed=0
[OK] Order_Details.csv         rows= 2156  dups_removed=9
[OK] Orders.csv                rows=  835  dups_removed=0
[OK] Products.csv              rows=   78  dups_removed=1
[OK] Region.csv                rows=    4  dups_removed=1
[OK] Shippers.csv              rows=    3  dups_removed=1
[OK] Suppliers.csv             rows=   30  dups_removed=2
[OK] Territories.csv           rows=   55  dups_removed=4

1. SMART FILL EFFECT REPORT (Orders Table)
     Table         Column  Missing_Before  Missing_After  Recovered
Orders.csv       ShipName               0              0          0
Orders.csv    ShipAddress               0              0          

In [14]:
import os
import pandas as pd

cleaned_dir = 'Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/cleaned_data'

print("="*70)
print("PHASE 3.2 - STEP 3 : FINALIZATION (Employees & Order_Details)")
print("="*70)

# =====================================================================
# (الف) Employees.csv  ->  ReportsTo یک FK است و missing معنادار است
# =====================================================================
emp_path = os.path.join(cleaned_dir, 'Employees.csv')
print("\n" + "-"*70)
print("FILE: Employees.csv")
print("-"*70)

df_emp = pd.read_csv(emp_path)

if 'ReportsTo' in df_emp.columns:
    n_miss = df_emp['ReportsTo'].isna().sum()
    print(f"  'ReportsTo' is a FOREIGN KEY (-> EmployeeID of manager).")
    print(f"  Missing in 'ReportsTo': {n_miss}")
    print("  Decision: KEEP as NULL. Missing means 'no manager' (e.g. top boss).")
    print("  -> NOT filled with median (would create a fake relation).")
else:
    print("  [i] 'ReportsTo' column not present.")

df_emp.to_csv(emp_path, index=False)
print(f"  Saved -> {emp_path}")

# =====================================================================
# (ب) Order_Details.csv  ->  اول حذف صفر/منفی، بعد پر کردن با میانه
# =====================================================================
od_path = os.path.join(cleaned_dir, 'Order_Details.csv')
print("\n" + "-"*70)
print("FILE: Order_Details.csv")
print("-"*70)

df_od = pd.read_csv(od_path)
before = len(df_od)

# گزارش تفکیکی صفر/منفی قبل از حذف
for col in ['UnitPrice', 'Quantity']:
    if col in df_od.columns:
        print(f"  {col}: (<0)={int((df_od[col] < 0).sum())} | "
              f"(==0)={int((df_od[col] == 0).sum())} | "
              f"(<=0)={int((df_od[col] <= 0).sum())}")

# 1) حذف ردیف‌های نامعتبر (صفر/منفی) - NaN ها حذف نمی‌شوند تا بعداً پر شوند
mask_invalid = (df_od['UnitPrice'] <= 0) | (df_od['Quantity'] <= 0)
df_od = df_od[~mask_invalid]
print(f"  Invalid rows removed (UnitPrice<=0 or Quantity<=0): "
      f"{before - len(df_od)}")

# 2) محاسبه میانه روی داده‌ی تمیز (بعد از حذف صفر/منفی)
for col in ['UnitPrice', 'Quantity']:
    n_miss = df_od[col].isna().sum()
    if n_miss > 0:
        med_v = df_od[col].median()
        df_od[col] = df_od[col].fillna(med_v)
        print(f"  '{col}': {n_miss} NaN filled with median={round(med_v, 2)} "
              f"(skewed -> median chosen)")
    else:
        print(f"  '{col}': no NaN to fill.")

# 3) Discount: صفر یک مقدار معتبر است -> missing با 0.0 پر می‌شود
if 'Discount' in df_od.columns:
    n_miss = df_od['Discount'].isna().sum()
    if n_miss > 0:
        df_od['Discount'] = df_od['Discount'].fillna(0.0)
        print(f"  'Discount': {n_miss} NaN filled with 0.0 (0 is valid here).")

df_od.to_csv(od_path, index=False)
print(f"  Saved -> {od_path}")

print("\n" + "="*70)
print("STEP 3 DONE - Phase 3.2 complete.")
print("="*70)


PHASE 3.2 - STEP 3 : FINALIZATION (Employees & Order_Details)

----------------------------------------------------------------------
FILE: Employees.csv
----------------------------------------------------------------------
  'ReportsTo' is a FOREIGN KEY (-> EmployeeID of manager).
  Missing in 'ReportsTo': 1
  Decision: KEEP as NULL. Missing means 'no manager' (e.g. top boss).
  -> NOT filled with median (would create a fake relation).
  Saved -> Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/cleaned_data\Employees.csv

----------------------------------------------------------------------
FILE: Order_Details.csv
----------------------------------------------------------------------
  UnitPrice: (<0)=4 | (==0)=2 | (<=0)=6
  Quantity: (<0)=5 | (==0)=5 | (<=0)=10
  Invalid rows removed (UnitPrice<=0 or Quantity<=0): 16
  'UnitPrice': 6 NaN filled with median=18.4 (skewed -> median chosen)
  'Quantity': 15 NaN filled with median=20.0 (skewed -> median chosen)
  'Discou

In [16]:
import pandas as pd
import os

# --- Unified base path (consistent with Phase 3.2 cells) ---
cleaned_dir = 'Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/cleaned_data'
report_output_path = 'Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/Data_Dictionary_Report.csv'

print("=" * 60)
print("        DATA DICTIONARY GENERATION REPORT")
print("=" * 60)

if not os.path.exists(cleaned_dir):
    print(f"[ERROR] Directory not found: {cleaned_dir}")
else:
    csv_files = sorted(f for f in os.listdir(cleaned_dir) if f.endswith('.csv'))
    dictionary_data = []

    for file_name in csv_files:
        file_path = os.path.join(cleaned_dir, file_name)
        try:
            df = pd.read_csv(file_path)
            table_name = file_name.replace('.csv', '')

            for col in df.columns:
                col_type = str(df[col].dtype)
                non_null_count = int(df[col].notnull().sum())

                valid_samples = df[col].dropna()
                sample_value = valid_samples.iloc[0] if not valid_samples.empty else "N/A"

                dictionary_data.append({
                    'Table_Name': table_name,
                    'Column_Name': col,
                    'Data_Type': col_type,
                    'Non_Null_Count': non_null_count,
                    'Sample_Value': str(sample_value)[:50]
                })

            print(f"  Scanned: {table_name:<28} cols={len(df.columns):<3} rows={len(df)}")

        except Exception as e:
            print(f"  [ERROR] Failed to scan {file_name}: {str(e)}")

    print("-" * 60)

    if dictionary_data:
        df_report = pd.DataFrame(dictionary_data)
        df_report.to_csv(report_output_path, index=False, encoding='utf-8-sig')

        print("SUMMARY")
        print(f"  Tables scanned : {len(csv_files)}")
        print(f"  Total columns  : {len(df_report)}")
        print("-" * 60)
        print("[STATUS] Data dictionary generated successfully.")
        print(f"[OUTPUT] {report_output_path}")

print("=" * 60)


        DATA DICTIONARY GENERATION REPORT
  Scanned: Categories                   cols=4   rows=8
  Scanned: CustomerCustomerDemo         cols=2   rows=0
  Scanned: CustomerDemographics         cols=2   rows=0
  Scanned: Customers                    cols=11  rows=91
  Scanned: EmployeeTerritories          cols=2   rows=51
  Scanned: Employees                    cols=18  rows=10
  Scanned: Order_Details                cols=5   rows=2140
  Scanned: Orders                       cols=14  rows=835
  Scanned: Products                     cols=10  rows=78
  Scanned: Region                       cols=2   rows=4
  Scanned: Shippers                     cols=3   rows=3
  Scanned: Suppliers                    cols=12  rows=30
  Scanned: Territories                  cols=3   rows=55
------------------------------------------------------------
SUMMARY
  Tables scanned : 13
  Total columns  : 88
------------------------------------------------------------
[STATUS] Data dictionary generated successful

In [ ]:
import pandas as pd
import os

cleaned_dir = 'Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/cleaned_data'

print("=" * 60)
print("        PHASE 3.3: DATA TYPE STANDARDIZATION")
print("=" * 60)

if not os.path.exists(cleaned_dir):
    print(f"[ERROR] Directory not found: {cleaned_dir}")
else:
    csv_files = sorted(f for f in os.listdir(cleaned_dir) if f.endswith('.csv'))

    date_columns      = ['BirthDate', 'HireDate', 'OrderDate', 'RequiredDate', 'ShippedDate']
    text_columns      = ['Phone', 'HomePhone', 'PostalCode', 'ShipPostalCode', 'Fax']
    financial_columns = ['UnitPrice', 'Freight', 'Discount', 'Quantity']

    for file_name in csv_files:
        file_path = os.path.join(cleaned_dir, file_name)
        try:
            df = pd.read_csv(file_path)
            table_name = file_name.replace('.csv', '')
            modified = False
            actions = []
            checks  = []

            # 1) Date columns -> standard datetime
            for col in date_columns:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                    actions.append(f"date    : {col}")
                    modified = True

            # 2) Text columns (phone / postal) -> verify + fix if read as numeric
            for col in text_columns:
                if col in df.columns:
                    was_numeric = pd.api.types.is_numeric_dtype(df[col])
                    mask = df[col].notnull()
                    df.loc[mask, col] = (
                        df.loc[mask, col].astype(str)
                        .str.replace(r'\.0$', '', regex=True)
                    )
                    if was_numeric:
                        checks.append(f"[FIX] '{col}' was read as numeric -> converted to text")
                    actions.append(f"text    : {col}")
                    modified = True

            # 3) Financial / numeric columns -> numeric
            for col in financial_columns:
                if col in df.columns and df[col].dtype == 'object':
                    df[col] = (
                        df[col].astype(str)
                        .str.replace('$', '', regex=False)
                        .str.replace(',', '', regex=False)
                        .str.strip()
                    )
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    checks.append(f"[FIX] '{col}' was stored as text -> converted to numeric")
                    actions.append(f"numeric : {col}")
                    modified = True

            # 4) ID columns -> check for non-numeric / null values
            id_cols = [c for c in df.columns if c.endswith('ID')]
            for col in id_cols:
                null_cnt = int(df[col].isnull().sum())
                non_numeric = pd.to_numeric(df[col], errors='coerce').isnull() & df[col].notnull()
                non_numeric_cnt = int(non_numeric.sum())
                if null_cnt > 0:
                    checks.append(f"[WARN] ID '{col}' has {null_cnt} null value(s)")
                if non_numeric_cnt > 0:
                    # CustomerID in Northwind is alphanumeric -> informational only
                    checks.append(f"[INFO] ID '{col}' has {non_numeric_cnt} non-numeric value(s)")

            print(f"  Table: {table_name}")
            if modified:
                for a in actions:
                    print(f"    - {a}")
                df.to_csv(file_path, index=False, encoding='utf-8-sig')
                print(f"    [STATUS] Updated and saved.")
            else:
                print(f"    [STATUS] No type changes required.")
            for c in checks:
                print(f"    {c}")
            print("-" * 60)

        except Exception as e:
            print(f"  [ERROR] Failed to process {file_name}: {str(e)}")
            print("-" * 60)

print("[DONE] Data types standardized and ready for Access / SQL import.")
print("=" * 60)


        PHASE 3.3: DATA TYPE STANDARDIZATION
  Table: Categories
    [STATUS] No type changes required.
------------------------------------------------------------
  Table: CustomerCustomerDemo
    [STATUS] No type changes required.
------------------------------------------------------------
  Table: CustomerDemographics
    [STATUS] No type changes required.
------------------------------------------------------------
  Table: Customers
    - text    : Phone
    - text    : PostalCode
    - text    : Fax
    [STATUS] Updated and saved.
    [INFO] ID 'CustomerID' has 91 non-numeric value(s)
------------------------------------------------------------
  Table: EmployeeTerritories
    [STATUS] No type changes required.
------------------------------------------------------------
  Table: Employees
    - date    : BirthDate
    - date    : HireDate
    - text    : HomePhone
    - text    : PostalCode
    [STATUS] Updated and saved.
-------------------------------------------------------

In [20]:
import pandas as pd
import os

cleaned_dir = 'Phase_1_Data Preprocessing_Python_R/Northwind Database - Cleaned/cleaned_data'

# Columns that are intentionally left NULL (meaningful absence)
intentional_nulls = {
    'Region', 'ShipRegion', 'Fax', 'Notes', 'Photo', 'PhotoPath',
    'Description', 'HomePage', 'TitleOfCourtesy', 'Extension',
    'ShippedDate', 'ReportsTo'
}

print("=" * 60)
print("        FINAL DATA QUALITY ASSURANCE (QA)")
print("=" * 60)

if not os.path.exists(cleaned_dir):
    print(f"[ERROR] Directory not found: {cleaned_dir}")
else:
    csv_files = sorted(f for f in os.listdir(cleaned_dir) if f.endswith('.csv'))

    for file_name in csv_files:
        file_path = os.path.join(cleaned_dir, file_name)
        try:
            df = pd.read_csv(file_path)
            table_name = file_name.replace('.csv', '')

            null_per_col = df.isnull().sum()
            total_nulls = int(null_per_col.sum())
            cols_with_nulls = [c for c in df.columns if null_per_col[c] > 0]

            print(f"  Table: {table_name}")
            print(f"    Rows    : {len(df)}")
            print(f"    Columns : {len(df.columns)}")

            if total_nulls == 0:
                print(f"    NULLs   : 0  (fully complete)")
            else:
                # Separate intentional vs unexpected
                unexpected = [c for c in cols_with_nulls if c not in intentional_nulls]
                print(f"    NULLs   : {total_nulls} total")
                if cols_with_nulls:
                    print(f"    NULL columns: {', '.join(cols_with_nulls)}")
                if unexpected:
                    print(f"    [REVIEW] Unexpected NULLs in: {', '.join(unexpected)}")
                else:
                    print(f"    [OK] All NULLs are intentional (meaningful absence).")
            print("-" * 60)

        except Exception as e:
            print(f"  [ERROR] Failed to check {file_name}: {str(e)}")
            print("-" * 60)

print("[DONE] QA completed. Data cleaning pipeline finished successfully.")
print("=" * 60)


        FINAL DATA QUALITY ASSURANCE (QA)
  Table: Categories
    Rows    : 8
    Columns : 4
    NULLs   : 4 total
    NULL columns: Description
    [OK] All NULLs are intentional (meaningful absence).
------------------------------------------------------------
  Table: CustomerCustomerDemo
    Rows    : 0
    Columns : 2
    NULLs   : 0  (fully complete)
------------------------------------------------------------
  Table: CustomerDemographics
    Rows    : 0
    Columns : 2
    NULLs   : 0  (fully complete)
------------------------------------------------------------
  Table: Customers
    Rows    : 91
    Columns : 11
    NULLs   : 90 total
    NULL columns: Region, Fax
    [OK] All NULLs are intentional (meaningful absence).
------------------------------------------------------------
  Table: EmployeeTerritories
    Rows    : 51
    Columns : 2
    NULLs   : 0  (fully complete)
------------------------------------------------------------
  Table: Employees
    Rows    : 10
    C